# 📈 Módulo 2: Series Temporales con Variable Exógena (Forecasting)
## Proyecto: AgroSmart Perú — Universidad / Software Inteligente

**Objetivo:** Pronosticar el precio en chacra al productor (S/ por kg) para limón sutil y arroz cáscara, incorporando retardos temporales ($t-1, t-2, t-3$) y el calentamiento del mar (**Índice ICEN**) como regresor exógeno.  
**Datos Reales:** Series mensuales de 2007 a 2026 del BCRP y MIDAGRI.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
os.makedirs('../models', exist_ok=True)
print("Librerías importadas correctamente.")


### 1. Carga de Series Históricas y Generación de Lags (Retardos)


In [ ]:
df = pd.read_csv('../data/tabla_maestra_agro_nino.csv')
df['fecha'] = pd.to_datetime(df['fecha'])
df_limon = df[df['cultivo'] == 'Limon'].sort_values('fecha').copy()

# Crear retardos temporales del precio y del ICEN
for lag in [1, 2, 3]:
    df_limon[f'precio_lag{lag}'] = df_limon['precio_chacra_soles_kg'].shift(lag)
    df_limon[f'icen_lag{lag}'] = df_limon['icen'].shift(lag)
    df_limon[f'lluvia_lag{lag}'] = df_limon['lluvia_mensual_mm'].shift(lag)

# Codificación trigonométrica de estacionalidad mensual
df_limon['mes_sin'] = np.sin(2 * np.pi * df_limon['mes'] / 12)
df_limon['mes_cos'] = np.cos(2 * np.pi * df_limon['mes'] / 12)

df_clean = df_limon.dropna().reset_index(drop=True)
print(f"Observaciones tras generar retardos: {len(df_clean)}")
df_clean[['fecha', 'precio_chacra_soles_kg', 'precio_lag1', 'icen', 'icen_lag1']].head()


### 2. Partición Cronológica Train/Test (Evitando Data Leakage)


In [ ]:
feat_cols = [c for c in df_clean.columns if 'lag' in c or 'mes_' in c or c in ['icen', 'anomalia_tsm']]
X = df_clean[feat_cols]
y = df_clean['precio_chacra_soles_kg']

# Usamos los últimos 36 meses como conjunto de prueba (Test ciego)
train_size = len(df_clean) - 36
X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]
fechas_test = df_clean['fecha'].iloc[train_size:]

print(f"Período Entrenamiento: {df_clean['fecha'].iloc[0].strftime('%Y-%m')} a {df_clean['fecha'].iloc[train_size-1].strftime('%Y-%m')}")
print(f"Período Evaluación (Test): {fechas_test.iloc[0].strftime('%Y-%m')} a {fechas_test.iloc[-1].strftime('%Y-%m')}")


### 3. Entrenamiento con Gradient Boosting Regressor


In [ ]:
model_ts = GradientBoostingRegressor(n_estimators=120, learning_rate=0.08, max_depth=4, random_state=42)
model_ts.fit(X_train, y_train)

y_pred = model_ts.predict(X_test)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("📊 Métricas de Evaluación de Pronóstico:")
print(f"• MAPE (Error Porcentual Absoluto Medio): {mape:.2f}%")
print(f"• RMSE: S/ {rmse:.2f} por kilogramo")


### 4. Gráfica de la Serie: Precio Real vs Pronóstico


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(fechas_test, y_test.values, 'o-', color='#1565c0', label='Precio Real BCRP/MIDAGRI (Soles/kg)')
plt.plot(fechas_test, y_pred, 's--', color='#d32f2f', lw=2, label=f'Pronóstico del Modelo (MAPE: {mape:.2f}%)')
plt.title('Evaluación de Pronóstico Temporal: Precios de Limón en Piura durante El Niño')
plt.ylabel('Soles por Kilogramo (S/)')
plt.legend()
plt.tight_layout()
plt.show()


### 5. Exportación del Modelo para la App y Simulación de Escenarios
El análisis anterior explica el modelo paso a paso. Para que el archivo `.pkl` que usa la aplicación tenga **exactamente** las variables que espera (`precio_chacra_soles_kg_lag1..3`, `icen_lag1..3`, `lluvia_lag1..3`, `mes_sin`, `mes_cos`, `icen`, `anomalia_tsm`), la exportación se hace con la misma clase de `src/timeseries.py` que carga `app.py`. Así hay una sola fuente de verdad entre el notebook y la aplicación.


In [ ]:
import sys
sys.path.append('..')
from src.timeseries import AgroTimeSeriesForecaster

forecaster = AgroTimeSeriesForecaster()
metricas_ts = forecaster.train_models()   # entrena Limón y Arroz y exporta models/ts_precio_*.pkl
pd.DataFrame(metricas_ts).T


In [ ]:
# Verificación: recargar desde disco (como lo hace la app) y proyectar escenarios ENFEN
app_forecaster = AgroTimeSeriesForecaster()
for escenario in ['Extraordinario (ICEN +2.6°C)', 'Normal / Neutro (ICEN +0.2°C)']:
    _, futuro = app_forecaster.forecast_future('Limon', months_ahead=6, escenario_nino=escenario)
    print(escenario, '->', futuro['precio_pronosticado_soles_kg'].tolist())
